<a href="https://colab.research.google.com/github/galang264/Data-Science/blob/main/TUGAS_WEEK_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DATA CLEANING - dataset_tugas_data_cleaning_OK**


In [ ]:
import pandas as pd
import numpy as np

1. LOAD DATA

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    PATH = list(up.keys())[0]
except ImportError:
    PATH = "/mnt/user-data/uploads/dataset_tugas_data_cleaning_OK.xlsx"

df_raw = pd.read_excel(PATH)
df = df_raw.copy()
print("Ukuran awal:", df.shape)

Saving dataset_tugas_data_cleaning_OK.xlsx to dataset_tugas_data_cleaning_OK (2).xlsx
Ukuran awal: (55, 5)


2. AUDIT (sebelum cleaning)

In [ ]:
audit_before = {
    "baris": len(df),
    "missing_usia": int(df["Usia"].isna().sum()),
    "missing_pendapatan": int(df["Pendapatan"].isna().sum()),
    "duplikat_persis": int(df.duplicated().sum()),
    "variasi_kota": int(df["Kota"].nunique()),
}
print(audit_before)
print(df["Kota"].value_counts())

{'baris': 55, 'missing_usia': 3, 'missing_pendapatan': 14, 'duplikat_persis': 5, 'variasi_kota': 7}
Kota
surabya       15
Jakarta       10
Bandung        9
jakrta         8
Yogyakarta     8
Surabaya       3
bandung        2
Name: count, dtype: int64


3. STANDARISASI TEKS

In [ ]:
# str.strip  -> hapus spasi berlebih  (Excel: TRIM)
# str.title  -> huruf kapital awal    (Excel: PROPER)
df["Nama"] = df["Nama"].str.strip().str.title()
df["Kota"] = df["Kota"].str.strip().str.lower()

# koreksi salah ketik (sama dengan tabel Referensi di Excel / VLOOKUP)
peta_kota = {
    "jakarta": "Jakarta", "jakrta": "Jakarta",
    "surabaya": "Surabaya", "surabya": "Surabaya",
    "bandung": "Bandung", "yogyakarta": "Yogyakarta",
}
df["Kota"] = df["Kota"].map(peta_kota)
assert df["Kota"].notna().all(), "Ada kota yang belum ada di peta_kota"

4. HAPUS DUPLIKAT

In [ ]:
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("Duplikat dihapus:", n_dup, "-> ukuran:", df.shape)

Duplikat dihapus: 5 -> ukuran: (50, 5)


5. MISSING VALUE: imputasi median

In [ ]:
median_usia = df["Usia"].median()
median_pendapatan = df["Pendapatan"].median()
df["Usia_Diimputasi"] = np.where(df["Usia"].isna(), "Ya", "Tidak")
df["Pendapatan_Diimputasi"] = np.where(df["Pendapatan"].isna(), "Ya", "Tidak")
df["Usia"] = df["Usia"].fillna(median_usia)
df["Pendapatan"] = df["Pendapatan"].fillna(median_pendapatan)
print("Median Usia:", median_usia, "| Median Pendapatan:", median_pendapatan)

Median Usia: 35.0 | Median Pendapatan: 5000000.0


6. OUTLIER

In [ ]:
def batas_iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    i = q3 - q1
    return q1 - 1.5 * i, q3 + 1.5 * i
for kol in ["Usia", "Pendapatan"]:
    lo, hi = batas_iqr(df[kol])
    n_out = int(((df[kol] < lo) | (df[kol] > hi)).sum())
    print(f"Outlier {kol}: {n_out} (batas {lo:,.0f} - {hi:,.0f})")

Outlier Usia: 0 (batas 15 - 55)
Outlier Pendapatan: 0 (batas 2,000,000 - 10,000,000)


7. TIPE DATA

In [ ]:
df["Usia"] = df["Usia"].round().astype(int)
df["Pendapatan"] = df["Pendapatan"].astype(int)

8. AUDIT SESUDAH & SIMPAN

In [ ]:
print("Missing setelah cleaning:\n", df.isna().sum())
print("Ukuran akhir:", df.shape)
print(df["Kota"].value_counts())
df.to_csv("dataset_bersih.csv", index=False)
df.to_excel("dataset_bersih.xlsx", index=False)
print(df.head())

Missing setelah cleaning:
 ID                       0
Nama                     0
Usia                     0
Kota                     0
Pendapatan               0
Usia_Diimputasi          0
Pendapatan_Diimputasi    0
dtype: int64
Ukuran akhir: (50, 7)
Kota
Jakarta       17
Surabaya      15
Bandung       10
Yogyakarta     8
Name: count, dtype: int64
   ID  Nama  Usia        Kota  Pendapatan Usia_Diimputasi  \
0   1  Cici    40  Yogyakarta     5000000           Tidak   
1   2  Cici    35     Jakarta     5000000           Tidak   
2   3  Gina    20     Jakarta     3000000           Tidak   
3   4  Hana    40    Surabaya     5000000           Tidak   
4   5  Hana    20    Surabaya     5000000           Tidak   

  Pendapatan_Diimputasi  
0                 Tidak  
1                 Tidak  
2                 Tidak  
3                    Ya  
4                    Ya  
